In [ ]:
from pathlib import Path
import json, sys, torch, kagglehub
print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), flush=True)
root = Path(kagglehub.competition_download('biohub-cell-tracking-during-development'))
print('Mounted:', root, flush=True)
print('Top folders:', [p.name for p in root.iterdir()], flush=True)
print('Train examples:', [p.name for p in (root/'train').iterdir()][:8], flush=True)


In [ ]:
%pip -q install zarr scipy scikit-learn
import zarr, numpy as np
print('Zarr:', zarr.__version__)
graphs = sorted((root/'train').glob('*.geff'))
embryos = sorted({p.stem.split('_')[0] for p in graphs})
print('Graphs:', len(graphs), 'Embryos:', embryos)
for embryo in embryos[:3]:
    p = next(p for p in graphs if p.stem.split('_')[0] == embryo)
    g = zarr.open_group(str(p), mode='r')
    ids=np.asarray(g['nodes/ids']); edges=np.asarray(g['edges/ids'])
    tt=np.asarray(g['nodes/props/t/values'])
    print(p.name, 'nodes', ids.shape, 'edges', edges.shape, 'times', np.unique(tt)[:12])


In [ ]:
# Stage 1: real annotated-edge learning. This is NOT end-to-end tracking accuracy.
import time, hashlib, copy
from collections import Counter, defaultdict
from sklearn.metrics import average_precision_score
from torch import nn
np.random.seed(42); torch.manual_seed(42); torch.set_num_threads(2)
out=Path('/kaggle/working/biohub_stage1'); out.mkdir(exist_ok=True)
scale=np.array([1.625,.40625,.40625])
train_paths=[p for p in graphs if p.stem.split('_')[0]=='44b6'][:12]
val_paths=[p for p in graphs if p.stem.split('_')[0]=='6bba'][:6]
assert train_paths and val_paths
assert {p.stem.split('_')[0] for p in train_paths}.isdisjoint({p.stem.split('_')[0] for p in val_paths})

def build(paths):
    features=[]; labels=[]; group_ids=[]; manifest=[]; total_targets=0
    for path in paths:
        g=zarr.open_group(str(path),mode='r')
        ids=np.asarray(g['nodes/ids']).astype(np.int64)
        coords=np.column_stack([np.asarray(g['nodes/props/'+k+'/values']) for k in ['t','z','y','x']]).astype(float)
        edges=np.asarray(g['edges/ids']).astype(np.int64)
        index={int(k):i for i,k in enumerate(ids)}
        incoming=defaultdict(list)
        for s,t in edges:
            if int(s) in index and int(t) in index: incoming[int(t)].append(int(s))
        manifest.append({'sample':path.stem,'nodes':len(ids),'edges':len(edges),'label_sha256':hashlib.sha256(ids.tobytes()+coords.tobytes()+edges.tobytes()).hexdigest()})
        for target,parents in incoming.items():
            if len(parents)!=1: continue
            j=index[target]; parent=parents[0]; i=index[parent]
            if coords[j,0]-coords[i,0]!=1: continue
            gid=total_targets; total_targets+=1
            candidates=np.flatnonzero(coords[:,0]==coords[i,0])
            delta=(coords[j,1:]-coords[candidates,1:])*scale
            dist=np.linalg.norm(delta,axis=1)
            order=np.argsort(dist)[:16]
            order=order[dist[order]<=20.]
            for rank,k in enumerate(order):
                d=delta[k]; r=dist[k]
                features.append(np.r_[d,np.abs(d),r,r*r,rank,len(candidates)])
                labels.append(int(ids[candidates[k]]==parent)); group_ids.append(gid)
    x=np.asarray(features,np.float32); y=np.asarray(labels,np.float32); groups=np.asarray(group_ids)
    assert len(x)>0 and y.sum()>0 and (1-y).sum()>0, 'Not enough positive/negative examples'
    return x,y,groups,total_targets,manifest

x,y,groups,nt,train_manifest=build(train_paths)
vx,vy,vg,vnt,val_manifest=build(val_paths)
mean=x.mean(0); std=x.std(0).clip(.001)
xt=torch.from_numpy((x-mean)/std); yt=torch.from_numpy(y)
vxt=torch.from_numpy((vx-mean)/std)
model=nn.Sequential(nn.Linear(x.shape[1],32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
optimizer=torch.optim.AdamW(model.parameters(),lr=.002)
loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((1-y).sum()/y.sum())))

def parent_accuracy(scores,truth,group,total):
    correct=0
    for gid in np.unique(group):
        ix=np.flatnonzero(group==gid)
        correct+=int(truth[ix[np.argmax(scores[ix])]]==1)
    return correct/max(total,1)

baseline={'average_precision':float(average_precision_score(vy,-vx[:,6])),
          'parent_accuracy':parent_accuracy(-vx[:,6],vy,vg,vnt)}
protocol={'seed':42,'training':train_manifest,'validation':val_manifest,
 'training_candidates':len(y),'training_positive_edges':int(y.sum()),'validation_candidates':len(vy),
 'validation_positive_edges':int(vy.sum()),'validation_eligible_targets':vnt,
 'candidate_parent_coverage':float(vy.sum()/vnt),'candidate_gate_um':20,'max_candidates':16,
 'features':['dz','dy','dx','abs_dz','abs_dy','abs_dx','distance','distance_squared','distance_rank','source_count'],
 'torch':torch.__version__,'device':'cpu','baseline':baseline,
 'limitations':['Uses ground-truth cell coordinates, not predicted detections','Validation embryo excluded from training, but used for checkpoint selection','Sparse labels: negatives are alternative parents of targets with one annotated parent','No image model or division classifier trained','Not official competition score, SEG, TRA, or world-best evidence']}
(out/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('TRAINING STARTED',json.dumps({k:protocol[k] for k in ['training_candidates','training_positive_edges','validation_candidates','candidate_parent_coverage','baseline']}),flush=True)
best=-1.; stale=0; history=[]; started=time.time()
for epoch in range(1,101):
    model.train(); batch_losses=[]
    for ix in torch.randperm(len(xt)).split(256):
        optimizer.zero_grad(); logits=model(xt[ix]).squeeze(1)
        loss=loss_fn(logits,yt[ix]); assert torch.isfinite(loss)
        loss.backward(); optimizer.step(); batch_losses.append(float(loss.detach()))
    model.eval()
    with torch.no_grad(): scores=model(vxt).squeeze(1).sigmoid().numpy()
    ap=float(average_precision_score(vy,scores))
    row={'epoch':epoch,'train_loss':float(np.mean(batch_losses)),'validation_average_precision':ap,
         'validation_parent_accuracy':parent_accuracy(scores,vy,vg,vnt)}
    history.append(row)
    if ap>best+1e-6:
        best=ap; stale=0; best_row=row.copy(); best_state=copy.deepcopy(model.state_dict())
        torch.save({'model':best_state,'mean':mean.tolist(),'std':std.tolist(),'metrics':row,'protocol':protocol},out/'best_linker.pt')
    else: stale+=1
    (out/'history.json').write_text(json.dumps(history,indent=2))
    (out/'status.json').write_text(json.dumps({'state':'training',**row}))
    if epoch==1 or epoch%10==0: print(json.dumps(row),flush=True)
    if epoch>=20 and stale>=15: break
result={'state':'completed','seconds':time.time()-started,'epochs_run':epoch,'best_validation':best_row,'baseline':baseline,
        'checkpoint_sha256':hashlib.sha256((out/'best_linker.pt').read_bytes()).hexdigest(),'limitations':protocol['limitations']}
(out/'status.json').write_text(json.dumps(result,indent=2))
print('TRAINING COMPLETED',json.dumps(result,indent=2),flush=True)


In [ ]:
# Full annotation linker: all training GEFFs; images are not used by this model.
from pathlib import Path
import json, time, hashlib
import numpy as np
import torch
from torch import nn
import zarr, kagglehub
from sklearn.metrics import average_precision_score
root = Path(kagglehub.competition_download('biohub-cell-tracking-during-development'))
paths = sorted((root/'train').glob('*.geff'))
assert len(paths) == 199, f'Expected 199 graphs, found {len(paths)}'
output = Path('/kaggle/working/biohub_full_annotations'); output.mkdir(exist_ok=True)
scale = np.array([1.625,.40625,.40625])
torch.set_num_threads(2)

def prepare(paths):
    features=[]; labels=[]; groups=[]; total=0; manifest=[]
    for p in paths:
        g=zarr.open_group(str(p),mode='r')
        ids=np.asarray(g['nodes/ids']); edges=np.asarray(g['edges/ids'])
        t=np.asarray(g['nodes/props/t/values'])
        xyz=np.column_stack([np.asarray(g[f'nodes/props/{k}/values']) for k in ['z','y','x']])*scale
        lookup={int(v):i for i,v in enumerate(ids)}; incoming={}
        for a,b in edges: incoming.setdefault(int(b),[]).append(int(a))
        bytime={v:np.flatnonzero(t==v) for v in np.unique(t)}
        start=total
        for target,parents in incoming.items():
            if len(parents)!=1: continue
            j=lookup[target]; parent=parents[0]; i=lookup[parent]
            if t[j]-t[i]!=1: continue
            gid=total; total+=1
            candidates=bytime[t[j]-1]
            delta=xyz[j]-xyz[candidates]; distance=np.linalg.norm(delta,axis=1)
            order=np.argsort(distance,kind='stable')[:16]
            for rank,k in enumerate(order):
                r=distance[k]
                if r>20: continue
                d=delta[k]
                features.append(np.r_[d,np.abs(d),r,r*r,rank,len(candidates)])
                labels.append(int(ids[candidates[k]]==parent)); groups.append(gid)
        manifest.append({'file':p.name,'nodes':len(ids),'edges':len(edges),'eligible_targets':total-start})
    return np.asarray(features,np.float32),np.asarray(labels,np.float32),np.asarray(groups),total,manifest

def evaluate(scores,y,groups,total):
    starts=np.r_[0,np.flatnonzero(np.diff(groups))+1]
    ends=np.r_[starts[1:],len(y)]
    correct=sum(int(y[a+np.argmax(scores[a:b])]) for a,b in zip(starts,ends))
    return {'average_precision':float(average_precision_score(y,scores)), 'parent_accuracy':correct/total,'correct_parents':correct,'eligible_targets':total,'candidate_coverage':float(y.sum()/total)}

def fit(x,y,name,epochs=27):
    torch.manual_seed(42); mean=x.mean(0); std=x.std(0).clip(.001)
    xt=torch.from_numpy((x-mean)/std); yt=torch.from_numpy(y)
    net=nn.Sequential(nn.Linear(10,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
    opt=torch.optim.AdamW(net.parameters(),lr=.002)
    lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((1-y).sum()/y.sum())))
    history=[]
    for epoch in range(1,epochs+1):
        net.train(); running=0
        for ix in torch.randperm(len(xt)).split(256):
            opt.zero_grad(); loss=lossfn(net(xt[ix]).squeeze(1),yt[ix]); assert torch.isfinite(loss)
            loss.backward(); opt.step(); running+=float(loss.detach())*len(ix)
        row={'epoch':epoch,'loss':running/len(y)}; history.append(row)
        if epoch==1 or epoch%10==0 or epoch==epochs: print(name,row,flush=True)
        (output/'status.json').write_text(json.dumps({'state':'training','phase':name,**row}))
    torch.save({'model':net.state_dict(),'mean':mean.tolist(),'std':std.tolist(),'epochs':epochs,'features':['dz','dy','dx','abs_dz','abs_dy','abs_dx','distance','distance_squared','distance_rank','source_count']},output/(name+'.pt'))
    (output/(name+'_history.json')).write_text(json.dumps(history,indent=2))
    return net,mean,std

started=time.time(); datasets={}
for embryo in sorted({p.stem.split('_')[0] for p in paths}):
    subset=[p for p in paths if p.stem.split('_')[0]==embryo]
    datasets[embryo]=prepare(subset)
    x,y,gr,total,manifest=datasets[embryo]
    print('LOADED',embryo,'graphs',len(subset),'candidates',len(y),'positive',int(y.sum()),flush=True)
report={'scope':'All 199 training GEFF annotations, not full image training','epochs':27,'epoch_choice':'Fixed from prior stage1; one fold overlaps prior development validation, so this is not an untouched test','seed':42,'device':'cpu','torch':torch.__version__,'gate_um':20,'max_candidates':16,'scale_zyx_um':scale.tolist(),'folds':[],'limitations':['Ground-truth coordinates, not predicted detections','Sparse annotation candidate-parent proxy, not official competition score','No detector or division classifier trained','Only two embryos; prior stage1 used both for development']}
for heldout in datasets:
    other=next(e for e in datasets if e!=heldout)
    x,y,_,_,_=datasets[other]; vx,vy,vg,vtotal,_=datasets[heldout]
    net,mean,std=fit(x,y,'holdout_'+heldout)
    net.eval()
    with torch.no_grad(): scores=net(torch.from_numpy((vx-mean)/std)).squeeze(1).numpy()
    fold={'training_embryo':other,'heldout_embryo':heldout,'learned':evaluate(scores,vy,vg,vtotal),'distance_baseline':evaluate(-vx[:,6],vy,vg,vtotal)}
    report['folds'].append(fold); print('FOLD RESULT',json.dumps(fold),flush=True)
    (output/'evaluation.json').write_text(json.dumps(report,indent=2))
x=np.concatenate([d[0] for d in datasets.values()]); y=np.concatenate([d[1] for d in datasets.values()])
fit(x,y,'final_all_annotations')
report.update({'state':'completed','graphs':len(paths),'training_candidates':len(y),'positive_edges':int(y.sum()),'seconds':time.time()-started,'manifest':[m for d in datasets.values() for m in d[4]],'checkpoint_sha256':hashlib.sha256((output/'final_all_annotations.pt').read_bytes()).hexdigest()})
(output/'evaluation.json').write_text(json.dumps(report,indent=2))
(output/'status.json').write_text(json.dumps({k:v for k,v in report.items() if k!='manifest'},indent=2))
print('FULL ANNOTATION TRAINING COMPLETED',json.dumps({k:v for k,v in report.items() if k!='manifest'},indent=2),flush=True)


In [ ]:
%pip -q install zarr scipy scikit-learn
from pathlib import Path
from collections import defaultdict
import copy, hashlib, json, time
import kagglehub, numpy as np, torch, zarr
from sklearn.metrics import average_precision_score
from torch import nn

# Stage 2 reads real image voxels for every one of the 199 annotated samples.
# Sparse missing annotations are never used as background labels.
root=Path(kagglehub.competition_download('biohub-cell-tracking-during-development'))
paths=sorted((root/'train').glob('*.geff'))
assert len(paths)==199, f'Expected 199 GEFF graphs, found {len(paths)}'
missing=[p.stem for p in paths if not (root/'train'/(p.stem+'.zarr')).exists()]
assert not missing, f'Missing paired image volumes: {missing[:5]}'
out=Path('/kaggle/working/biohub_image_linker_full'); out.mkdir(exist_ok=True)
scale=np.array([1.625,.40625,.40625],np.float32)
np.random.seed(42); torch.manual_seed(42); torch.set_num_threads(2)

def local_image_features(frame, point):
    z,y,x=np.rint(point).astype(int)
    z=int(np.clip(z,0,frame.shape[0]-1)); y=int(np.clip(y,0,frame.shape[1]-1)); x=int(np.clip(x,0,frame.shape[2]-1))
    patch=frame[max(0,z-1):min(frame.shape[0],z+2),max(0,y-3):min(frame.shape[1],y+4),max(0,x-3):min(frame.shape[2],x+4)].astype(np.float32)
    sampled=frame[::4,::8,::8].astype(np.float32)
    lo,hi=np.percentile(sampled,[1,99.8]); denom=max(float(hi-lo),1.0)
    patch=np.clip((patch-lo)/denom,0,2)
    return np.array([patch.mean(),patch.std(),patch.max(),np.percentile(patch,75),float((patch>.5).mean())],np.float32)

def build_image_pairs(sample_paths,tag):
    features=[]; labels=[]; groups=[]; total=0; manifest=[]; started=time.time()
    for sample_i,p in enumerate(sample_paths,1):
        g=zarr.open_group(str(p),mode='r')
        ids=np.asarray(g['nodes/ids']).astype(np.int64)
        t=np.asarray(g['nodes/props/t/values']).astype(int)
        xyz=np.column_stack([np.asarray(g[f'nodes/props/{k}/values']) for k in ['z','y','x']]).astype(np.float32)
        edges=np.asarray(g['edges/ids']).astype(np.int64)
        image=zarr.open(str(root/'train'/(p.stem+'.zarr')),mode='r')['0']
        image_feat=np.zeros((len(ids),5),np.float32)
        for tv in np.unique(t):
            if tv<0 or tv>=image.shape[0]: continue
            frame=np.asarray(image[int(tv)])
            for idx in np.flatnonzero(t==tv): image_feat[idx]=local_image_features(frame,xyz[idx])
        lookup={int(v):i for i,v in enumerate(ids)}; incoming=defaultdict(list)
        for s,u in edges:
            if int(s) in lookup and int(u) in lookup: incoming[int(u)].append(int(s))
        bytime={tv:np.flatnonzero(t==tv) for tv in np.unique(t)}; before=total
        for target,parents in incoming.items():
            if len(parents)!=1: continue
            j=lookup[target]; parent=parents[0]; i=lookup[parent]
            if t[j]-t[i]!=1: continue
            gid=total; total+=1; candidates=bytime[t[i]]
            delta=(xyz[j]-xyz[candidates])*scale; distance=np.linalg.norm(delta,axis=1)
            order=np.argsort(distance,kind='stable')[:16]
            for rank,k in enumerate(order):
                r=float(distance[k])
                if r>20: continue
                d=delta[k]; src=image_feat[candidates[k]]; dst=image_feat[j]
                features.append(np.r_[d,np.abs(d),r,r*r,rank,len(candidates),src,dst,np.abs(src-dst)])
                labels.append(int(ids[candidates[k]]==parent)); groups.append(gid)
        manifest.append({'sample':p.stem,'nodes':len(ids),'edges':len(edges),'eligible_targets':total-before,'image_shape':list(image.shape)})
        (out/'status.json').write_text(json.dumps({'state':'extracting','split':tag,'samples_done':sample_i,'samples_total':len(sample_paths),'candidates':len(labels),'seconds':time.time()-started}))
        if sample_i==1 or sample_i%10==0 or sample_i==len(sample_paths): print('EXTRACT',tag,sample_i,'/',len(sample_paths),'candidates',len(labels),flush=True)
    return np.asarray(features,np.float32),np.asarray(labels,np.float32),np.asarray(groups,np.int64),total,manifest

def metrics(scores,y,groups,total):
    correct=0
    for gid in np.unique(groups):
        ix=np.flatnonzero(groups==gid); correct+=int(y[ix[np.argmax(scores[ix])]]==1)
    return {'average_precision':float(average_precision_score(y,scores)),'parent_accuracy':correct/max(total,1),'correct_parents':correct,'eligible_targets':total,'candidate_coverage':float(y.sum()/max(total,1))}

def fit(x,y,name,epochs=15):
    mean=x.mean(0); std=x.std(0).clip(.001); xt=torch.from_numpy((x-mean)/std); yt=torch.from_numpy(y)
    torch.manual_seed(42); net=nn.Sequential(nn.Linear(x.shape[1],64),nn.ReLU(),nn.Dropout(.05),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
    opt=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=1e-4); lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((1-y).sum()/y.sum())))
    history=[]
    for epoch in range(1,epochs+1):
        net.train(); running=0.
        for ix in torch.randperm(len(xt)).split(1024):
            opt.zero_grad(); loss=lossfn(net(xt[ix]).squeeze(1),yt[ix]); loss.backward(); opt.step(); running+=float(loss.detach())*len(ix)
        row={'epoch':epoch,'loss':running/len(y)}; history.append(row)
        (out/'status.json').write_text(json.dumps({'state':'training','phase':name,**row}))
        print('TRAIN',name,row,flush=True)
    torch.save({'model':net.state_dict(),'mean':mean.tolist(),'std':std.tolist(),'input_features':x.shape[1],'epochs':epochs},out/(name+'.pt'))
    (out/(name+'_history.json')).write_text(json.dumps(history,indent=2)); return net,mean,std

started=time.time(); datasets={}
for embryo in sorted({p.stem.split('_')[0] for p in paths}):
    datasets[embryo]=build_image_pairs([p for p in paths if p.stem.split('_')[0]==embryo],embryo)
report={'state':'training','scope':'All 199 paired GEFF graphs and image volumes; local patches at annotated nodes','seed':42,'device':'cuda' if torch.cuda.is_available() else 'cpu','features':['10 geometry','5 source-image','5 target-image','5 absolute-image-difference'],'folds':[],'limitations':['Ground-truth coordinates seed image patches; this is an image-aware linker, not an image detector','Sparse labels: alternatives are candidate parents, never unlabeled voxels as background','Only two embryo IDs; folds are embryo-disjoint but not an independent competition test','No official competition metric or submission']}
for heldout in datasets:
    train_emb=next(e for e in datasets if e!=heldout); x,y,_,_,_=datasets[train_emb]; vx,vy,vg,vt,_=datasets[heldout]
    net,mean,std=fit(x,y,'image_holdout_'+heldout); net.eval()
    with torch.no_grad(): score=net(torch.from_numpy((vx-mean)/std)).squeeze(1).numpy()
    fold={'training_embryo':train_emb,'heldout_embryo':heldout,'image_model':metrics(score,vy,vg,vt),'distance_baseline':metrics(-vx[:,6],vy,vg,vt)}
    report['folds'].append(fold); (out/'evaluation.json').write_text(json.dumps(report,indent=2)); print('FOLD',json.dumps(fold),flush=True)
x=np.concatenate([d[0] for d in datasets.values()]); y=np.concatenate([d[1] for d in datasets.values()])
fit(x,y,'image_final_all_199')
report.update({'state':'completed','samples':199,'training_candidates':len(y),'positive_edges':int(y.sum()),'seconds':time.time()-started,'manifest':[m for d in datasets.values() for m in d[4]],'checkpoint_sha256':hashlib.sha256((out/'image_final_all_199.pt').read_bytes()).hexdigest()})
(out/'evaluation.json').write_text(json.dumps(report,indent=2)); (out/'status.json').write_text(json.dumps({k:v for k,v in report.items() if k!='manifest'},indent=2))
print('FULL IMAGE-AWARE TRAINING COMPLETED',json.dumps({k:v for k,v in report.items() if k!='manifest'},indent=2),flush=True)
